Тема: Застосування методів колаборативної фільтрації для генерації рекомендацій.

Завдання 1. Для заданих відомих тем у сфері комп’ютерних наук, застосуйте метод колаборативної фільтрації для того, щоб виділити основні рекомендації, які найточніше відповідають запитам інтересів користувачів. 
Завдання 2. Згенеруйте ефективні рекомендації для різних користувачів, та визначіть їх тему сфери інтересів.

In [3]:
from collections import Counter, defaultdict
import numpy as np
import math

# Вхідні дані - інтереси користувачів
users_interests = [
    ["Hadoop", "Big Data", "HBase", "Java", "Spark", "Storm", "Cassandra"],
    ["NoSQL", "MongoDB", "Cassandra", "HBase", "Postgres"],
    ["Python", "scikit-learn", "scipy", "numpy", "statsmodels", "pandas"],
    ["R", "Python", "statistics", "regression", "probability"],
    ["machine learning", "regression", "decision trees", "libsvm"],
    ["Python", "R", "Java", "C++", "Haskell", "programming languages"],
    ["statistics", "probability", "mathematics", "theory"],
    ["machine learning", "scikit-learn", "Mahout", "neural networks"],
    ["neural networks", "deep learning", "Big Data", "artificial intelligence"],
    ["Hadoop", "Java", "MapReduce", "Big Data"],
    ["statistics", "R", "statsmodels"],
    ["C++", "deep learning", "artificial intelligence", "probability"],
    ["pandas", "R", "Python"],
    ["databases", "HBase", "Postgres", "MySQL", "MongoDB"],
    ["libsvm", "regression", "support vector machines"]
]

# Визначення найпопулярніших тем
popular_interests = Counter(
    interest for user_interests in users_interests for interest in user_interests
).most_common()

def most_popular_new_interests(user_interests, max_results=5):
    suggestions = [(interest, frequency)
                   for interest, frequency in popular_interests if interest not in user_interests]
    return suggestions[:max_results]

# Косинусна подібність

def cosine_similarity(v, w):
    return np.dot(v, w) / math.sqrt(np.dot(v, v) * np.dot(w, w))

# Унікальні інтереси
unique_interests = sorted(list({interest for user in users_interests for interest in user}))

def make_user_interest_vector(user_interests):
    return [1 if interest in user_interests else 0 for interest in unique_interests]

# Матриця інтересів
user_interest_matrix = list(map(make_user_interest_vector, users_interests))

# Матриця подібності користувачів
user_similarities = [[cosine_similarity(user_i, user_j) for user_j in user_interest_matrix] 
                     for user_i in user_interest_matrix]

def most_similar_users_to(user_id):
    pairs = [(other_user_id, similarity) for other_user_id, similarity in enumerate(user_similarities[user_id])
             if user_id != other_user_id and similarity > 0]
    return sorted(pairs, key=lambda x: x[1], reverse=True)

def user_based_suggestions(user_id, include_current_interests=False):
    suggestions = defaultdict(float)
    for other_user_id, similarity in most_similar_users_to(user_id):
        for interest in users_interests[other_user_id]:
            suggestions[interest] += similarity
    
    suggestions = sorted(suggestions.items(), key=lambda x: x[1], reverse=True)
    if include_current_interests:
        return suggestions
    else:
        return [(suggestion, weight) for suggestion, weight in suggestions if suggestion not in users_interests[user_id]]

# Транспонована матриця
interest_user_matrix = [[user_interest_vector[j] for user_interest_vector in user_interest_matrix]
                        for j, _ in enumerate(unique_interests)]

# Матриця подібності тем
interest_similarities = [[cosine_similarity(interest_i, interest_j) for interest_j in interest_user_matrix]
                         for interest_i in interest_user_matrix]

def most_similar_interests_to(interest):
    interest_id = unique_interests.index(interest)
    similarities = interest_similarities[interest_id]
    pairs = [(unique_interests[i], similarity) for i, similarity in enumerate(similarities)
             if i != interest_id and similarity > 0]
    return sorted(pairs, key=lambda x: x[1], reverse=True)

def item_based_suggestions(user_id):
    suggestions = defaultdict(float)
    user_interest_vector = user_interest_matrix[user_id]
    for interest_id, is_interested in enumerate(user_interest_vector):
        if is_interested == 1:
            for interest, similarity in most_similar_interests_to(unique_interests[interest_id]):
                suggestions[interest] += similarity
    return sorted(suggestions.items(), key=lambda x: x[1], reverse=True)

# Генерація рекомендацій для користувачів
for user_id in range(len(users_interests)):
    print(f"Рекомендації для користувача {user_id}: {user_based_suggestions(user_id)}")
    print(f"Схожі інтереси для користувача {user_id}: {item_based_suggestions(user_id)}")
    print("-"*50)

Рекомендації для користувача 0: [('MapReduce', 0.5669467095138409), ('MongoDB', 0.50709255283711), ('Postgres', 0.50709255283711), ('NoSQL', 0.3380617018914066), ('neural networks', 0.1889822365046136), ('deep learning', 0.1889822365046136), ('artificial intelligence', 0.1889822365046136), ('databases', 0.1690308509457033), ('MySQL', 0.1690308509457033), ('Python', 0.1543033499620919), ('R', 0.1543033499620919), ('C++', 0.1543033499620919), ('Haskell', 0.1543033499620919), ('programming languages', 0.1543033499620919)]
Схожі інтереси для користувача 0: [('Spark', 4.146264369941973), ('Storm', 4.146264369941973), ('Hadoop', 3.9554550146924106), ('Cassandra', 3.547206724228547), ('Java', 3.3794454097708404), ('Big Data', 3.3794454097708404), ('HBase', 3.0461120764375074), ('MapReduce', 1.861807319565799), ('MongoDB', 1.3164965809277263), ('Postgres', 1.3164965809277263), ('NoSQL', 1.2844570503761732), ('MySQL', 0.5773502691896258), ('databases', 0.5773502691896258), ('Haskell', 0.5773502